In [24]:
import subprocess
import time
import itertools
import os
import sys
import pandas as pd
import glob
from datetime import datetime

# ================= 設定區 =================
'''
dic = {
    'batch_size': [1000],
    'epoch': [1000],
    #'layer': [4, 3, 2],
    'layer': [2],
    #'hidden': [32, 16],
    'hidden': [32],
    'data_version': [58],            # 確保版本號與訓練腳本一致
    #'lr': [0.003, 0.001, 0.03, 0.01],
    'lr': [0.03],
    'column': ['acceleration_X,acceleration_Y,acceleration_Z,gyroscope_X,gyroscope_Y,gyroscope_Z'],
    'folds': [2],
    # ==== 保留網格參數槽（相容介面，此處的 target_version 將作為 CORAL 映射源） ====
    'lambda_dann': [1.0],            
    'target_version': ['58/special_data'], 
}'''

dic = {
    'batch_size': [1000],
    'epoch': [1000],
    'layer': [4, 3, 2],
    'hidden': [32, 16],
    'data_version': [58],            # 確保版本號與訓練腳本一致
    'lr': [0.003, 0.001, 0.03, 0.01],
    'column': ['acceleration_X,acceleration_Y,acceleration_Z,gyroscope_X,gyroscope_Y,gyroscope_Z'],
    'folds': [1, 2, 3, 4, 5],
    # ==== 保留網格參數槽（相容介面，此處的 target_version 將作為 CORAL 映射源） ====
    'lambda_dann': [1.0],            
    'target_version': ['58/special_data'], 
}

MAX_CONCURRENT_JOBS = 5  # 訓練與推論時的併發數（A100 上跑標準 GRU 速度極快）
TRAIN_SCRIPT = "train.py" 
TEST_SCRIPT = "test.py"   
OUTPUT_LOG_DIR = "./test_log"
# =========================================

def get_combinations(params):
    keys = list(params.keys())
    values = list(params.values())
    for combo in itertools.product(*values):
        yield dict(zip(keys, combo))

def run_phase(phase_name, script_name, combinations, max_jobs):
    print(f"\n=== 開始執行階段: {phase_name} ===")
    total_jobs = len(combinations)
    running_processes = []
    
    for i, p in enumerate(combinations):
        cmd = [
            'python3', script_name,
            f'--batch_size={p["batch_size"]}',
            f'--epoch={p["epoch"]}',
            f'--layer={p["layer"]}',
            f'--hidden={p["hidden"]}',
            f'--data_version={p["data_version"]}',
            f'--lr={p["lr"]}',
            f'--column={p["column"]}',
            f'--fold={p["folds"]}', 
            f'--lambda_dann={p["lambda_dann"]}',
            f'--target_version={p["target_version"]}'
        ]
        
        # 保持標準錯誤輸出可見，方便排查環境問題
        proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL)
        running_processes.append(proc)
        
        # 進度顯示
        if i % 10 == 0:
            print(f"[{phase_name}] 進度: {i}/{total_jobs} (Running: {len(running_processes)})")

        # 控管併發數
        while len(running_processes) >= max_jobs:
            running_processes = [proc for proc in running_processes if proc.poll() is None]
            if len(running_processes) >= max_jobs:
                time.sleep(1)

    # 等待最後一批完成
    for proc in running_processes:
        proc.wait()
    print(f"=== {phase_name} 階段完成 ===\n")

def collect_results():
    print(f"=== 正在從 {OUTPUT_LOG_DIR} 彙整最新測試報告 ===")
    
    # 💡 【關鍵修改 1】：更新搜尋規則，CORAL 測試產出的成果都會包含 _CORAL 關鍵字
    result_files = glob.glob(os.path.join(OUTPUT_LOG_DIR, "*_CORAL_result.csv"))
    
    # 若找不到，退回相容掃描 test_log 底下所有的 csv
    if not result_files:
        result_files = glob.glob(os.path.join(OUTPUT_LOG_DIR, "*.csv"))
        if not result_files or any("Final_Report" in f for f in result_files):
            print(f"在 {OUTPUT_LOG_DIR} 找不到任何有效的測試結果檔案。")
            return

    all_dfs = []
    for f in result_files:
        if "Final_Report" in f: continue # 避免重複讀取歷史總報告
        try:
            df = pd.read_csv(f)
            if not df.empty:
                all_dfs.append(df)
        except Exception as e:
            print(f"讀取 {f} 失敗: {e}")
    
    if all_dfs:
        final_df = pd.concat(all_dfs, ignore_index=True)
        
        if "macro_f1" in final_df.columns:
            final_df = final_df.sort_values(by="macro_f1", ascending=False)
            
        # 💡 【關鍵修改 2】：將產出的總報告名稱更名為 CORAL 版本
        out_name = "Final_Report_CORAL_v58.csv"
        final_df.to_csv(out_name, index=False)
        
        print("-" * 50)
        print(f"報告整合完成！共匯總 {len(all_dfs)} 筆測試數據。")
        print(f"最終報告已產出: {out_name}")
        print("-" * 50)
        print("Top 5 最佳模型表現 (前置統計學對齊優化):")
        
        # 💡 【關鍵修改 3】：重新精簡終端機印出的欄位，專注於跨使用者適應後的各受試者疲勞預估精準度
        try:
            print(final_df.head(5)[['run_name', 'acc', 'macro_f1', 'mAP', 'f1_notTired', 'f1_Tired', 'f1_Other']])
        except KeyError:
            print(final_df.head(5))
            
    else:
        print("沒有有效的數據可以合併。")

def main():
    start_time = datetime.now()
    combinations = list(get_combinations(dic)) 
    
    # 階段 1: 訓練（初次換架構，強烈建議取消註解先跑一次 Training）
    #run_phase("Training", TRAIN_SCRIPT, combinations, max_jobs=MAX_CONCURRENT_JOBS)
    
    # 階段 2: 測試
    run_phase("Testing", TEST_SCRIPT, combinations, max_jobs=MAX_CONCURRENT_JOBS)
    
    # 階段 3: 彙整
    collect_results()

    end_time = datetime.now()
    print(f"總耗時: {end_time - start_time}")

if __name__ == "__main__":
    main()


=== 開始執行階段: Testing ===
[Testing] 進度: 0/1 (Running: 1)
=== Testing 階段完成 ===

=== 正在從 ./test_log 彙整最新測試報告 ===
--------------------------------------------------
報告整合完成！共匯總 1 筆測試數據。
最終報告已產出: Final_Report_CORAL_v58.csv
--------------------------------------------------
Top 5 最佳模型表現 (前置統計學對齊優化):
                                            run_name       acc  macro_f1  \
0  fold2_layer_2_hidden_32_lr_0.03_dv_58_col_acce...  0.877926  0.668701   

        mAP  f1_notTired  f1_Tired  f1_Other  
0  0.685616     0.859922  0.173913  0.972268  
總耗時: 0:00:02.933837
